<a href="https://colab.research.google.com/github/Farrukh776/flyrank-ai/blob/main/work/notebooks/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Farrukh776/flyrank-ai/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

This lane maps to Ranking / Scoring. The question "which pages should be reviewed first?" is a "which ones first?" question — per the framing guide's task-type table, that's ranking/scoring, with a priority score as the target and Precision@K as the metric (not plain classification accuracy, since what matters is getting the top of the list right, not every row).

In [ ]:
import os, subprocess, sys

if "google.colab" in sys.modules and not os.path.exists("flyrank-ai"):
    subprocess.run(["git", "clone", "https://github.com/Farrukh776/flyrank-ai.git"], check=True)
if os.path.basename(os.getcwd()) != "flyrank-ai":
    os.chdir("flyrank-ai")

import pandas as pd
df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
print("Loaded:", df.shape)

print("Total pages:", len(df))
print("Pages currently visible (impressions_90d > 0):", (df["impressions_90d"] > 0).sum())

Loaded: (30000, 44)
Total pages: 30000
Pages currently visible (impressions_90d > 0): 30000


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

The starter pipeline's target is is_declining_label = (trend_direction == "down"). This is a proxy label, not a true future outcome — it's a bucket computed from the current 90-day window's own trend_pct, not something observed in a later, separate time window. That's an honest limitation the lane guide itself flags: a stronger capstone target would be "prior 90 days of features → decline over the next 30 days." For this framing exercise I'm using the proxy since that's what the starter data supports, but I'm naming it as a proxy on purpose, not treating it as ground truth.

In [ ]:
df["is_declining_label"] = (df["trend_direction"].str.lower() == "down").astype(int)
print(df["trend_direction"].value_counts())
print(f"\nDeclining rate: {df['is_declining_label'].mean():.1%}")

trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

Declining rate: 54.2%


## 3. Success metric

*One metric you can defend. What number means 'good'?*

Precision@50 — of the top 50 pages the ranking flags, what fraction are truly declining? This matches how the output is actually used: an editor with limited time reviews a fixed-size list, so it matters more that the top of the list is right than that every single page in the dataset is classified correctly. The starter pipeline already shows this concretely: baseline rule = 0.240, random forest = 0.740.

In [ ]:
print("Baseline rule  Precision@50: 0.240  (~12 of top 50 correct)")
print("Random forest  Precision@50: 0.740  (~37 of top 50 correct)")

Baseline rule  Precision@50: 0.240  (~12 of top 50 correct)
Random forest  Precision@50: 0.740  (~37 of top 50 correct)


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

One row = one pseudonymized content item (page) belonging to one client, described by its trailing-90-day search and engagement metrics

In [ ]:
cols = ["content_id", "client_id", "impressions_90d", "sessions_90d",
        "avg_position", "ctr", "content_age_days", "days_since_last_update",
        "trend_direction", "is_declining_label"]
df[cols].head(10)

,content_id,client_id,impressions_90d,sessions_90d,avg_position,ctr,content_age_days,days_since_last_update,trend_direction,is_declining_label
0,content_304f48230142,client_f369cb89fc,3803,17,10.6,0.76,187,20,down,1
1,content_a1fb4e703a9e,client_4e07408562,15320,9,20.3,0.05,445,25,down,1
2,content_9aa793d4d895,client_7f2253d7e2,12581,11,36.5,0.09,141,20,down,1
3,content_331d6c4de07b,client_19581e27de,11751,78,6.2,0.49,463,22,stable,0
4,content_d99b7a2d90ca,client_3fdba35f04,19140,145,44.0,0.13,263,14,down,1
5,content_d4084a4bc775,client_f369cb89fc,3970,5,8.5,0.03,147,20,down,1
6,content_9a34b442b552,client_8722616204,20,1,7.0,0.00,90,20,down,1
7,content_a63219c6e95a,client_19581e27de,1724,28,21.2,0.06,445,22,stable,0
8,content_5e6c160719bc,client_6208ef0f77,32574,68,46.0,0.09,90,20,down,1
9,content_c27558df2b0c,client_19581e27de,1240,3,4.9,0.16,257,104,down,1


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A page's decline risk depends on several signals interacting at once — staleness, visibility, position, CTR, engagement — not any single threshold. The starter's hand-written rule (stale AND visible) only catches 24% of its top-50 picks correctly, while a random forest trained on the same signals catches 74%. That gap is exactly what the lane guide describes: "the pattern is real but too messy to write by hand — many signals, tangled, shifting over time." A human could write one or two if-statements, but not the nonlinear combination of six-plus signals that the model found.

In [ ]:
import pandas as pd
comparison = pd.DataFrame({
    "method": ["baseline rule", "random forest"],
    "precision_at_50": [0.240, 0.740]
})
comparison

,method,precision_at_50
0,baseline rule,0.24
1,random forest,0.74


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.